In [1]:
import pandas as pd
import numpy as np
import json

# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_csv('data_features.csv')
N = len(df)

def freq(count):
    """Format as 'N (X.X%)'."""
    return f"{count} ({100 * count / N:.1f}%)"

def mean_sd(series):
    """Format as 'X.XX (X.XX)'."""
    return f"{series.mean():.2f} ({series.std():.2f})"

rows = []

# ── AGE ──────────────────────────────────────────────────────────────────────
rows.append({'Characteristic': 'Age', 'Category': '', 'Frequency (%)': '', 'Mean (SD)': mean_sd(df['age'])})

# ── GENDER ───────────────────────────────────────────────────────────────────
gender_cats = ['Male', 'Female', 'Transgender', 'Other/Prefer not to say']
for i, cat in enumerate(gender_cats):
    cnt = (df['gender'] == cat).sum()
    rows.append({
        'Characteristic': 'Gender' if i == 0 else '',
        'Category': cat,
        'Frequency (%)': freq(cnt),
        'Mean (SD)': '',
    })

# ── ETHNICITY ────────────────────────────────────────────────────────────────
eth_cats = [
    ('Hispanic (any race)',    df['ethnicity'].str.contains('Hispanic', na=False) & ~df['ethnicity'].str.contains('Not Hispanic', na=False)),
    ('Not Hispanic or Latino', df['ethnicity'].str.contains('Not Hispanic', na=False)),
]
for i, (cat, mask) in enumerate(eth_cats):
    rows.append({
        'Characteristic': 'Ethnicity' if i == 0 else '',
        'Category': cat,
        'Frequency (%)': freq(mask.sum()),
        'Mean (SD)': '',
    })

# ── RACE ─────────────────────────────────────────────────────────────────────
def parse_race(val):
    if pd.isna(val):
        return []
    try:
        parsed = json.loads(val.replace("'", '"'))
        return parsed if isinstance(parsed, list) else [str(parsed)]
    except (json.JSONDecodeError, AttributeError):
        return [str(val)]

race_exploded = df['race'].apply(parse_race).explode()
race_counts = race_exploded.value_counts()
for i, (cat, cnt) in enumerate(race_counts.items()):
    rows.append({
        'Characteristic': 'Race' if i == 0 else '',
        'Category': cat,
        'Frequency (%)': freq(cnt),
        'Mean (SD)': '',
    })

# ── EDUCATION ────────────────────────────────────────────────────────────────
edu_order = ['High school only', 'Some college', "Bachelor's degree", "Master's degree", 'Doctoral degree']
edu_counts = df['education'].value_counts()
for i, cat in enumerate(edu_order):
    cnt = edu_counts.get(cat, 0)
    rows.append({
        'Characteristic': 'Education' if i == 0 else '',
        'Category': cat,
        'Frequency (%)': freq(cnt),
        'Mean (SD)': '',
    })

# ── HEALTH LITERACY ──────────────────────────────────────────────────────────
hl = df['health_literacy']
for i, val in enumerate(range(1, 6)):
    if val < 5:
        cnt = int(((hl >= val) & (hl < val + 1)).sum())
    else:
        cnt = int((hl >= 5).sum())
    rows.append({
        'Characteristic': 'Health Literacy' if i == 0 else '',
        'Category': str(val),
        'Frequency (%)': freq(cnt),
        'Mean (SD)': mean_sd(hl) if i == 0 else '',
    })

# ── SUBJECTIVE NUMERACY SCALE ───────────────────────────────────────────────
sn = df['subjective_numeracy']
sn_bins = [
    (1, 2, '1-1.99'), (2, 3, '2-2.99'), (3, 4, '3-3.99'),
    (4, 5, '4-4.99'), (5, 6, '5-5.99'), (6, 7, '6'),
]
for i, (lo, hi, label) in enumerate(sn_bins):
    if label == '6':
        cnt = int((sn >= 6).sum())
    else:
        cnt = int(((sn >= lo) & (sn < hi)).sum())
    rows.append({
        'Characteristic': 'Subjective Numeracy Scale' if i == 0 else '',
        'Category': label,
        'Frequency (%)': freq(cnt),
        'Mean (SD)': mean_sd(sn) if i == 0 else '',
    })

# ── GRAPHICAL LITERACY SCALE ────────────────────────────────────────────────
gl = df['graphical_literacy_score']
for i, val in enumerate(range(0, 8)):
    cnt = int((gl == val).sum())
    rows.append({
        'Characteristic': 'Graphical Literacy Scale' if i == 0 else '',
        'Category': str(val),
        'Frequency (%)': freq(cnt),
        'Mean (SD)': mean_sd(gl) if i == 0 else '',
    })

# ── FAMILIARITY WITH MEDICAL TESTS ──────────────────────────────────────────
fam = df['familiarity']
for i, val in enumerate(range(1, 6)):
    cnt = int((fam == val).sum())
    rows.append({
        'Characteristic': 'Familiarity with Medical Tests' if i == 0 else '',
        'Category': str(val),
        'Frequency (%)': freq(cnt),
        'Mean (SD)': mean_sd(fam) if i == 0 else '',
    })

# ── Build and save table ────────────────────────────────────────────────────
table1 = pd.DataFrame(rows, columns=['Characteristic', 'Category', 'Frequency (%)', 'Mean (SD)'])
table1.to_csv('table1_demographics.csv', index=False)
print(table1.to_string(index=False))
print("\nSaved table1_demographics.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data_features.csv'